# Model Merging with MergeKit

## Beyond LoRA Merging

In notebook 28, we merged LoRA adapters back into base weights. **Model merging** is different:
it combines **entire fine-tuned models** to create a new model with combined capabilities.

This is a major trend in the open-source community. Most top models on the Open LLM Leaderboard
are merges — combining a math-specialized model with a coding model with a chat model, for example.

**Key algorithms:**

| Method | How | When |
|--------|-----|------|
| Linear | W_merged = Σ(αᵢ · Wᵢ) | Quick blending, often works |
| SLERP | Spherical interpolation | Smooth 2-model blend |
| TIES | Prune + elect + merge | Multiple models, avoids interference |
| DARE | Random drop + rescale | Sparsifies before merging |
| Passthrough | Stack layers from different models | Depth extension |

**MergeKit** is the standard tool — supports all of the above, runs on CPU (no GPU needed for merging).

In [ ]:
!pip install mergekit -q 2>&1 | tail -3

In [ ]:
# MergeKit uses YAML configs — the entire merge is declarically specified
import yaml, subprocess, os, json

# Example: SLERP merge of two fine-tuned variants
# In this demo we merge the base model with itself at different interpolation
# to show the mechanics (in practice you'd merge two different fine-tuned models)
slerp_config = {
    "merge_method": "slerp",
    "base_model": "Qwen/Qwen3-0.6B",
    "models": [
        {"model": "Qwen/Qwen3-0.6B", "parameters": {"t": 0}},   # base (t=0)
        {"model": "Qwen/Qwen3-0.6B", "parameters": {"t": 1}},   # target (t=1)
    ],
    "parameters": {"t": 0.5},    # interpolation point 0=base, 1=target, 0.5=midpoint
    "dtype": "float16",
    "tokenizer_source": "base",
}

with open("/tmp/merge_config.yaml", "w") as f:
    yaml.dump(slerp_config, f)

print("SLERP merge config:")
print(yaml.dump(slerp_config))
print("\nIn practice, you'd use two DIFFERENT fine-tuned models:")
print("  Model A: fine-tuned on math problems")
print("  Model B: fine-tuned on code generation")
print("  Result: a model that handles both!")

In [ ]:
# Run MergeKit CLI
result = subprocess.run(
    ["mergekit-yaml", "/tmp/merge_config.yaml", "/tmp/merged-model", "--cuda", "--lazy-unpickle"],
    capture_output=True, text=True, timeout=300
)
if result.returncode == 0:
    print("✅ Merge complete!")
    files = os.listdir("/tmp/merged-model")
    print(f"Output files: {files}")
else:
    print(f"MergeKit output:\n{result.stdout[-500:]}\n{result.stderr[-500:]}")

In [ ]:
# TIES merging — for combining multiple models
# TIES: Trim (prune near-zero deltas) → Elect (resolve sign conflicts) → Merge
ties_config = {
    "merge_method": "ties",
    "base_model": "Qwen/Qwen3-0.6B",
    "models": [
        {"model": "Qwen/Qwen3-0.6B"},
        {"model": "Qwen/Qwen3-0.6B"},
    ],
    "parameters": {
        "density": 0.5,   # keep top 50% of weights by magnitude
        "weight": 1.0,    # equal weighting for both models
    },
    "dtype": "float16",
}

print("TIES config explains the algorithm:")
print("  1. Trim: zero out weights below density threshold in task vectors")
print("  2. Elect: for conflicting signs, pick majority sign")  
print("  3. Merge: add elected task vectors to base model")
print()
print("DARE (Drop And REscale) is similar but uses random dropping:")
print("  - Drop random p% of delta weights (not magnitude-based)")
print("  - Rescale remaining by 1/(1-p) to preserve expected norm")
print("  - More aggressive sparsification, better for models with large deltas")
print()
print("Both TIES and DARE help when merging many models to avoid parameter interference")

## Practical Merging Tips

**When merging works well:**
- Models fine-tuned on complementary tasks (math + code, chat + reasoning)
- Models with the same base architecture and tokenizer
- Models fine-tuned with LoRA (smaller deltas → less interference)

**When merging fails:**
- Very different base models or different training distributions
- Models with very large task vectors (catastrophic forgetting)
- Too many models (>4-5 starts degrading)

**No GPU needed!** MergeKit runs model merging on CPU — you can merge 70B models on a machine 
with no GPU as long as you have enough RAM (2× model size in float16).

**Community recipes:** Check HuggingFace Hub for `mergekit` tagged repos — thousands of recipes 
for popular model families.